# BOCOFAC – Product Performance: Simple Linear Regression (Yearly, Monthly, Weekly)

Data source: `BOCOFAC_SALES_DATA_2026_FINAL.xlsx`, sheet **Product Performance**.

Model: **Y = a + bX**, where **Y** = Total Sales (₱) and **X** = time (year / month number / week number).

**How to use in Google Colab:** Runtime ▸ Run all, then upload the Excel file when the *Choose Files* button appears.

## Step 1. Install and import libraries

In [1]:
!pip install -q statsmodels openpyxl

import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import statsmodels.api as sm
import openpyxl
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)

## Step 2. Upload the Excel file

When the **Choose Files** button appears, select **BOCOFAC_SALES_DATA_2026_FINAL.xlsx** (the Excel file — *not* this .ipynb notebook).

In [2]:
import os
FILE = None
try:
    from google.colab import files
    while FILE is None:
        uploaded = files.upload()
        excel = [f for f in uploaded if f.lower().endswith(('.xlsx', '.xlsm'))]
        if excel:
            FILE = excel[0]
        else:
            print('⚠ That is not an Excel file:', list(uploaded.keys()))
            print('  Please upload BOCOFAC_SALES_DATA_2026_FINAL.xlsx (ends in .xlsx).')
except ImportError:
    FILE = 'BOCOFAC_SALES_DATA_2026_FINAL.xlsx'   # running outside Colab
print('Using file:', FILE)

Using file: BOCOFAC_SALES_DATA_2026_FINAL.xlsx


## Step 3. Read the Yearly, Monthly and Weekly tables from the Product Performance sheet

In [3]:
wb = openpyxl.load_workbook(FILE, data_only=True)

def from_sheet(title):
    rows = list(wb['Product Performance'].iter_rows(values_only=True))
    start = next(i for i, r in enumerate(rows) if r[0] == title)
    header = list(rows[start + 1])
    data = []
    for r in rows[start + 2:]:
        if r[0] is None or r[0] == 'TOTAL':
            break
        data.append(r)
    df = pd.DataFrame(data, columns=header)
    df = df.rename(columns={header[0]: 'Period'})
    return df[['Period', 'From', 'To', 'Most In-Demand Product', 'Qty Sold', 'Total Sales (₱)']]

def from_sales_data(level):
    # fallback: build the same table directly from the Sales Data sheet
    s = pd.read_excel(FILE, sheet_name='Sales Data')
    s['OrderedDate'] = pd.to_datetime(s['OrderedDate'])
    if level == 'Y':
        key = s['OrderedDate'].dt.year
        idx = sorted(key.unique())
    elif level == 'M':
        key = s['OrderedDate'].dt.to_period('M')
        idx = pd.period_range(key.min(), key.max(), freq='M')
    else:
        key = s['OrderedDate'] - pd.to_timedelta(s['OrderedDate'].dt.weekday, unit='D')
        idx = pd.date_range(key.min(), key.max(), freq='7D')
    sales = s.groupby(key)['TotalAmount'].sum().reindex(idx, fill_value=0)
    qty = s.groupby([key, 'Product'])['Quantity'].sum().unstack(fill_value=0).reindex(idx, fill_value=0)
    top = qty.idxmax(axis=1).where(qty.max(axis=1) > 0, 'No orders')
    return pd.DataFrame({'Period': [str(i) for i in idx], 'From': None, 'To': None,
                         'Most In-Demand Product': top.values, 'Qty Sold': qty.max(axis=1).values,
                         'Total Sales (₱)': sales.values})

data = {}
for lvl, title in [('Y', 'A. Yearly'), ('M', 'B. Monthly'), ('W', 'C. Weekly')]:
    try:
        data[lvl] = from_sheet(title)
        src = 'Product Performance sheet'
    except Exception as e:
        data[lvl] = from_sales_data(lvl)
        src = 'Sales Data sheet'
    print(f'{title[3:]:8s}: {len(data[lvl]):3d} periods, total sales = ₱{data[lvl]["Total Sales (₱)"].sum():,.0f}  (read from {src})')

yearly, monthly, weekly = data['Y'], data['M'], data['W']
yearly['X']  = yearly['Period'].astype(int)         # X = year
monthly['X'] = np.arange(1, len(monthly) + 1)       # X = month no. (Nov 2021 = 1)
weekly['X']  = np.arange(1, len(weekly) + 1)        # X = week no. (week of Nov 1, 2021 = 1)

Yearly  :   6 periods, total sales = ₱2,111,386  (read from Product Performance sheet)
Monthly :  58 periods, total sales = ₱2,111,386  (read from Product Performance sheet)
Weekly  : 253 periods, total sales = ₱2,111,386  (read from Product Performance sheet)


### 3a. Yearly data

In [4]:
print(yearly[['X', 'Period', 'Most In-Demand Product', 'Qty Sold', 'Total Sales (₱)']].to_string(index=False))

   X  Period Most In-Demand Product  Qty Sold  Total Sales (₱)
2021    2021   Unscreened Coco Peat       102            72791
2022    2022     Screened Coco Peat       614           449084
2023    2023   Unscreened Coco Peat       552           437498
2024    2024     Screened Coco Peat       483           433426
2025    2025     Screened Coco Peat       509           437365
2026    2026     Screened Coco Peat       293           281222


### 3b. Monthly data

In [5]:
print(monthly[['X', 'Period', 'Most In-Demand Product', 'Qty Sold', 'Total Sales (₱)']].to_string(index=False))

 X   Period   Most In-Demand Product  Qty Sold  Total Sales (₱)
 1 Nov 2021     Unscreened Coco Peat        79            43245
 2 Dec 2021       Organic Fertilizer        39            29546
 3 Jan 2022       Coconut Rope (12m)        28            31078
 4 Feb 2022       Screened Coco Peat        64            35314
 5 Mar 2022       Screened Coco Peat        38            35385
 6 Apr 2022       Screened Coco Peat        71            40220
 7 May 2022       Organic Fertilizer        59            33999
 8 Jun 2022       Screened Coco Peat        63            39566
 9 Jul 2022       Screened Coco Peat        79            39992
10 Aug 2022     Unscreened Coco Peat        69            42660
11 Sep 2022       Organic Fertilizer        38            28734
12 Oct 2022       Screened Coco Peat        53            39584
13 Nov 2022       Screened Coco Peat        71            43640
14 Dec 2022     Unscreened Coco Peat        61            38912
15 Jan 2023       Screened Coco Peat    

### 3c. Weekly data

In [6]:
print(weekly[['X', 'Period', 'From', 'To', 'Most In-Demand Product', 'Qty Sold', 'Total Sales (₱)']].to_string(index=False))

  X   Period       From         To      Most In-Demand Product  Qty Sold  Total Sales (₱)
  1   Week 1 2021-11-01 2021-11-07        Unscreened Coco Peat        26             9180
  2   Week 2 2021-11-08 2021-11-14        Unscreened Coco Peat        17             8640
  3   Week 3 2021-11-15 2021-11-21        Unscreened Coco Peat        28            12110
  4   Week 4 2021-11-22 2021-11-28          Screened Coco Peat        23            10850
  5   Week 5 2021-11-29 2021-12-05    Coconut Shell Briquettes        16             9493
  6   Week 6 2021-12-06 2021-12-12    Coconut Shell Briquettes        21             6829
  7   Week 7 2021-12-13 2021-12-19          Organic Fertilizer        29             5625
  8   Week 8 2021-12-20 2021-12-26          Coconut Rope (12m)        12             7370
  9   Week 9 2021-12-27 2022-01-02                Coconut Coir        17             5269
 10  Week 10 2022-01-03 2022-01-09  Large Woven Rattan Storage        10             9013
 11  Week 

## Step 4. Regression function

In [7]:
def strength(r):
    r = abs(r)
    return ('very weak' if r < 0.2 else 'weak' if r < 0.4 else 'moderate' if r < 0.6
            else 'strong' if r < 0.8 else 'very strong')

def peso(v, unit=''):
    s = f'₱{abs(v):,.2f}{unit}'
    return '-' + s if v < 0 else s

def run_regression(df, title, unit, x_pred, pred_label, obs_label, what):
    model = sm.OLS(df['Total Sales (₱)'].astype(float), sm.add_constant(df['X'].astype(float))).fit()
    n, r2 = int(model.nobs), model.rsquared
    r, b, a = np.sqrt(r2), model.params['X'], model.params['const']
    F, p = model.fvalue, model.f_pvalue
    pred = a + b * x_pred
    d1, d2 = int(model.df_model), int(model.df_resid)

    rows = [('Number of observations', f'{n} {obs_label}'), ('R', f'{r:.3f}'), ('R²', f'{r2:.3f}'),
            ('Adjusted R²', f'{model.rsquared_adj:.3f}'), ('F-value', f'{F:.3f}'), ('p-value', f'{p:.3f}'),
            ('Regression coefficient (B)', peso(b, unit)), ('Constant', peso(a)), (pred_label, peso(pred))]
    w = max(len(k) for k, _ in rows) + 4
    line = '-' * (w + 22)
    print(title)
    print(line)
    print(f"{'Statistic':<{w}}Result")
    print(line)
    for k, v in rows:
        print(f'{k:<{w}}{v}')
    print(line)
    sig = 'statistically significant' if p < 0.05 else 'not statistically significant'
    print(f"\nInterpretation: The {what} regression shows a {strength(r)} {'positive' if b > 0 else 'negative'} "
          f"relationship between time and product performance (R = {r:.3f}). The model explains approximately "
          f"{r2*100:.1f}% of the variation in {what} sales, and the regression is {sig}, F({d1},{d2}) = {F:.3f}, "
          f"p = {p:.3f}. The estimated linear trend {'increases' if b > 0 else 'decreases'} by approximately "
          f"{peso(abs(b), unit)}. Using the regression equation, the projected product performance "
          f"{pred_label.replace('Prediction ', '')} is approximately {peso(pred)}.\n")
    return model

## Step 5. Yearly Linear Regression – Product Performance

In [8]:
m_year = run_regression(yearly, 'Yearly Linear Regression – Product Performance', '',
                        2027, 'Prediction for 2027', 'years', 'yearly')

Yearly Linear Regression – Product Performance
----------------------------------------------------
Statistic                     Result
----------------------------------------------------
Number of observations        6 years
R                             0.356
R²                            0.126
Adjusted R²                   -0.092
F-value                       0.579
p-value                       0.489
Regression coefficient (B)    ₱28,655.03
Constant                      -₱57,631,552.65
Prediction for 2027           ₱452,190.27
----------------------------------------------------

Interpretation: The yearly regression shows a weak positive relationship between time and product performance (R = 0.356). The model explains approximately 12.6% of the variation in yearly sales, and the regression is not statistically significant, F(1,4) = 0.579, p = 0.489. The estimated linear trend increases by approximately ₱28,655.03. Using the regression equation, the projected product performance f

## Step 6. Monthly Linear Regression – Product Performance

In [9]:
m_month = run_regression(monthly, 'Monthly Linear Regression – Product Performance', '/month',
                         len(monthly) + 1, 'Prediction for September 2026', 'months', 'monthly')

Monthly Linear Regression – Product Performance
-------------------------------------------------------
Statistic                        Result
-------------------------------------------------------
Number of observations           58 months
R                                0.085
R²                               0.007
Adjusted R²                      -0.011
F-value                          0.403
p-value                          0.528
Regression coefficient (B)       -₱23.52/month
Constant                         ₱37,096.91
Prediction for September 2026    ₱35,709.50
-------------------------------------------------------

Interpretation: The monthly regression shows a very weak negative relationship between time and product performance (R = 0.085). The model explains approximately 0.7% of the variation in monthly sales, and the regression is not statistically significant, F(1,56) = 0.403, p = 0.528. The estimated linear trend decreases by approximately ₱23.52/month. Using the regressi

## Step 7. Weekly Linear Regression – Product Performance

In [10]:
m_week = run_regression(weekly, 'Weekly Linear Regression – Product Performance', '/week',
                        len(weekly) + 1, 'Prediction for week of September 7, 2026', 'weeks', 'weekly')

Weekly Linear Regression – Product Performance
------------------------------------------------------------------
Statistic                                   Result
------------------------------------------------------------------
Number of observations                      253 weeks
R                                           0.069
R²                                          0.005
Adjusted R²                                 0.001
F-value                                     1.187
p-value                                     0.277
Regression coefficient (B)                  -₱2.05/week
Constant                                    ₱8,605.66
Prediction for week of September 7, 2026    ₱8,085.14
------------------------------------------------------------------

Interpretation: The weekly regression shows a very weak negative relationship between time and product performance (R = 0.069). The model explains approximately 0.5% of the variation in weekly sales, and the regression is not statis

## Step 8. Summary of the three models

In [11]:
summary = pd.DataFrame([{
    'Model': name, 'n': int(m.nobs), 'R': f'{np.sqrt(m.rsquared):.3f}', 'R²': f'{m.rsquared:.3f}',
    'Adj. R²': f'{m.rsquared_adj:.3f}', 'F-value': f'{m.fvalue:.3f}', 'p-value': f'{m.f_pvalue:.3f}',
    'B': f'{m.params["X"]:,.2f}', 'Constant': f'{m.params["const"]:,.2f}'}
    for name, m in [('Yearly', m_year), ('Monthly', m_month), ('Weekly', m_week)]])
print(summary.to_string(index=False))

  Model   n     R    R² Adj. R² F-value p-value         B       Constant
 Yearly   6 0.356 0.126  -0.092   0.579   0.489 28,655.03 -57,631,552.65
Monthly  58 0.085 0.007  -0.011   0.403   0.528    -23.52      37,096.91
 Weekly 253 0.069 0.005   0.001   1.187   0.277     -2.05       8,605.66


## Step 9. Most in-demand product – count of periods each product was #1

In [12]:
for name, df in [('Yearly', yearly), ('Monthly', monthly), ('Weekly', weekly)]:
    counts = df.loc[df['Most In-Demand Product'] != 'No orders', 'Most In-Demand Product'].value_counts()
    print(f'{name} – number of {name.lower()[:-2] if name != "Monthly" else "month"}s each product was the most in demand:')
    print(counts.to_string(), '\n')

Yearly – number of years each product was the most in demand:
Most In-Demand Product
Screened Coco Peat      4
Unscreened Coco Peat    2 

Monthly – number of months each product was the most in demand:
Most In-Demand Product
Screened Coco Peat          22
Unscreened Coco Peat        18
Organic Fertilizer          10
Coconut Rope (12m)           3
Coconut Coir                 2
Coconut Shell Briquettes     2
Coconut Husk Pole            1 

Weekly – number of weeks each product was the most in demand:
Most In-Demand Product
Screened Coco Peat             74
Unscreened Coco Peat           64
Organic Fertilizer             44
Coconut Shell Briquettes       30
Coconut Rope (12m)             14
Coconut Coir                   11
Coconut Husk Pole              11
Large Woven Rattan Storage      2
Plant Stand                     2
Woven Rattan Storage Basket     1 



## Step 10. Accuracy of the predictions

* **MAE** (Mean Absolute Error) – average peso difference between actual and predicted sales.
* **RMSE** (Root Mean Squared Error) – like MAE but gives more weight to big misses.
* **MAPE** (Mean Absolute Percentage Error) – average % error. **Accuracy = 100% − MAPE.**

Guide for MAPE (Lewis, 1982): below 10% = highly accurate, 10–20% = good, 20–50% = reasonable, above 50% = inaccurate.

**A. Fit accuracy** – the model is compared with the same data it was built from.
**B. Hold-out test** – the model is rebuilt *without* the last 12 months / last 52 weeks, then used to predict those periods, and the predictions are compared with what actually happened.

In [13]:
def accuracy(actual, predicted):
    actual, predicted = np.asarray(actual, float), np.asarray(predicted, float)
    err = actual - predicted
    nz = actual != 0
    mape = np.mean(np.abs(err[nz] / actual[nz])) * 100
    level = ('Highly accurate' if mape < 10 else 'Good' if mape < 20 else 'Reasonable' if mape < 50 else 'Inaccurate')
    return {'MAE (₱)': f'{np.mean(np.abs(err)):,.2f}', 'RMSE (₱)': f'{np.sqrt(np.mean(err**2)):,.2f}',
            'MAPE': f'{mape:.2f}%', 'Accuracy': f'{100 - mape:.2f}%', 'Rating': level}

rows = []
for name, df, m in [('Yearly', yearly, m_year), ('Monthly', monthly, m_month), ('Weekly', weekly, m_week)]:
    pred = m.predict(sm.add_constant(df['X'].astype(float)))
    rows.append({'Model': name, 'Test': 'A. Fit (all data)', **accuracy(df['Total Sales (₱)'], pred)})

for name, df, hold in [('Monthly', monthly, 12), ('Weekly', weekly, 52)]:
    train, test = df.iloc[:-hold], df.iloc[-hold:]
    mt = sm.OLS(train['Total Sales (₱)'].astype(float), sm.add_constant(train['X'].astype(float))).fit()
    pred = mt.params['const'] + mt.params['X'] * test['X'].astype(float)
    rows.append({'Model': name, 'Test': f'B. Hold-out (last {hold})', **accuracy(test['Total Sales (₱)'], pred)})

print(pd.DataFrame(rows).to_string(index=False))
print()
print('Note: Yearly has only 6 points, so no hold-out test is done. Its MAPE is high because 2021 (Nov–Dec only)')
print('and 2026 (Jan–Aug only) are partial years, so their totals are much lower than a full year.')

  Model                  Test    MAE (₱)   RMSE (₱)   MAPE Accuracy     Rating
 Yearly     A. Fit (all data) 116,594.11 128,614.59 69.15%   30.85% Inaccurate
Monthly     A. Fit (all data)   3,700.38   4,640.53 10.97%   89.03%       Good
 Weekly     A. Fit (all data)   1,709.28   2,176.17 27.43%   72.57% Reasonable
Monthly B. Hold-out (last 12)   3,940.92   4,835.82 11.32%   88.68%       Good
 Weekly B. Hold-out (last 52)   1,795.02   2,325.07 45.03%   54.97% Reasonable

Note: Yearly has only 6 points, so no hold-out test is done. Its MAPE is high because 2021 (Nov–Dec only)
and 2026 (Jan–Aug only) are partial years, so their totals are much lower than a full year.


## (Optional) Full OLS output

In [14]:
for name, m in [('YEARLY', m_year), ('MONTHLY', m_month), ('WEEKLY', m_week)]:
    print('=' * 30, name, '=' * 30)
    print(m.summary())

============================== YEARLY ==============================


                            OLS Regression Results                            
Dep. Variable:        Total Sales (₱)   R-squared:                       0.126
Model:                            OLS   Adj. R-squared:                 -0.092
Method:                 Least Squares   F-statistic:                    0.5791
Date:                Sat, 03 Oct 2026   Prob (F-statistic):              0.489
Time:                        22:32:45   Log-Likelihood:                -79.101
No. Observations:                   6   AIC:                             162.2
Df Residuals:                       4   BIC:                             161.8
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -5.763e+07   7.62e+07     -0.756      0.4